# 04 · Modeling — approach and first results (DSO1.1 next-product propensity)
**Phase 3 (Skill 3 – Modeling)** · NextDeal for BH Assurance · *status: baseline done, tuning in progress*

**Question.** For a company, *which product should the agent propose next?*

**Framing.** Leave-one-out episodes built in notebook 03: one product the company owns is hidden,
the model sees the profile + the other products, and must **rank all products not yet held**.
The hidden product should come out on top. This is an offline portfolio-reconstruction proxy. Production sees the complete current portfolio; future purchases, conversion and causal uplift are not validated here. The catalogue has 46 products, but only 34 have positive examples; the v1 classifier cannot learn the 12 candidate-only classes.

**Business-aligned metrics.** An advisor (or the agent) pitches at most 3 products, so we measure
- **HitRate@3** – share of episodes where the hidden product is in the top-3 (main metric);
- **HitRate@1, NDCG@3, MRR** – quality of the order;
- **Long-tail HitRate@3** – same, restricted to products *outside* the 3 best-sellers (motor ×2,
  multi-risk). These are the non-obvious products that really grow the portfolio;
- **Coverage** – how many different products the system ever recommends (personalisation).

**Validation.** 5-fold **GroupKFold by company** (a company never appears in train and test),
repeated with 3 shuffles → mean ± std over 15 folds. Encoders are fitted inside each fold. Learned imputation mappings from notebook 03 were fitted before splitting; contract-history inputs also require an availability audit. These limitations qualify the estimate.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from src.utils import (PROCESSED, ROOT, set_style, save, subtitle, PALETTE, INK, INK_2, INK_3, GRID,
                       rank_of_true, ranking_report, cluster_bootstrap_ci)
from src.features import build_preprocessor

warnings.filterwarnings("ignore")
set_style()
BLUE, ORANGE, GREEN = PALETTE[0], PALETTE[1], PALETTE[2]

episodes = pd.read_csv(PROCESSED / "pm_episodes.csv").fillna({"visible_portfolio": ""})
catalogue = json.loads((PROCESSED / "catalogue.json").read_text())
PRODUCTS = catalogue["products"]
IDX = {p: i for i, p in enumerate(PRODUCTS)}
y = episodes["hidden_product"].map(IDX).to_numpy()
groups = episodes["REF_pm"].to_numpy()

VISIBLE = np.zeros((len(episodes), len(PRODUCTS)), dtype=bool)
for r, v in enumerate(episodes["visible_portfolio"]):
    for p in filter(None, v.split(";")):
        VISIBLE[r, IDX[p]] = True

TOP3 = set(pd.Series(y).value_counts().index[:3])
print(f"{len(episodes)} episodes · {episodes['REF_pm'].nunique()} companies · {len(PRODUCTS)} products")
print("best-sellers excluded from the long-tail metric:", [PRODUCTS[i] for i in TOP3])

440 episodes · 299 companies · 46 products
best-sellers excluded from the long-tail metric: ['MULTIRISQUES PROFESSIONNELLES CENTRALISE', 'AUTOMOBILE', 'AUTOMOBILE II']


## 1. Candidate models

| Family | Model | Idea |
|---|---|---|
| Naive | **Random** | expected HitRate@3 = 3 / (number of candidates) |
| Business rule | **Popularity** | always propose the best-sellers not yet held (what a generic campaign does) |
| Business rule | **Segment popularity** | best-sellers *of the client's business category* |
| Recommender | **Item co-occurrence** | "companies holding X also hold Y" (item-based collaborative filtering) |
| ML | **Multinomial logistic regression** | profile + portfolio → probability of each product |
| ML | **Random forest** | non-linear interactions sector × portfolio |
| ML | **Gradient boosting (HistGB)** | boosted trees; stand-in for CatBoost used in the prototype |

In [2]:
def train_pop(tr):
    return np.bincount(y[tr], minlength=len(PRODUCTS)).astype(float)


def popularity(tr, te):
    return np.tile(train_pop(tr), (len(te), 1))


def segment_popularity(tr, te, alpha=2.0):
    pop = train_pop(tr)
    prior = pop / pop.sum()
    cats = episodes["category"].to_numpy()
    out = np.zeros((len(te), len(PRODUCTS)))
    for j, i in enumerate(te):
        same = tr[cats[tr] == cats[i]]
        out[j] = np.bincount(y[same], minlength=len(PRODUCTS)) + alpha * prior
    return out


def cooccurrence(tr, te, lam=0.05):
    pop = train_pop(tr)
    C = np.zeros((len(PRODUCTS), len(PRODUCTS)))
    for i in tr:
        for v in np.flatnonzero(VISIBLE[i]):
            C[v, y[i]] += 1
    row = C.sum(axis=1, keepdims=True)
    P_next = np.divide(C, row, out=np.zeros_like(C), where=row > 0)
    return VISIBLE[te].astype(float) @ P_next + lam * pop / pop.max()


def sklearn_model(estimator):
    def fit_predict(tr, te):
        pipe = make_pipeline(build_preprocessor(PRODUCTS), estimator)
        pipe.fit(episodes.iloc[tr], y[tr])
        out = np.zeros((len(te), len(PRODUCTS)))
        out[:, pipe.classes_] = pipe.predict_proba(episodes.iloc[te])
        pop = train_pop(tr)
        return out + 1e-6 * pop / pop.max()          # tiny tie-break by popularity
    return fit_predict


MODELS = {
    "Popularity": popularity,
    "Segment popularity": segment_popularity,
    "Co-occurrence": cooccurrence,
    "Logistic regression": sklearn_model(LogisticRegression(C=0.3, max_iter=3000)),
    "Random forest": sklearn_model(RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                                          random_state=0, n_jobs=-1)),
    "Gradient boosting": sklearn_model(HistGradientBoostingClassifier(max_iter=120, learning_rate=0.05, max_depth=3,
                                                                      l2_regularization=1.0, random_state=0)),
}
try:
    from catboost import CatBoostClassifier   # optional: used in the prototype, not installed everywhere
    MODELS["CatBoost"] = sklearn_model(CatBoostClassifier(iterations=300, depth=4, verbose=0, random_seed=0))
except ImportError:
    print("catboost not installed → skipped (pip install catboost to add it)")

catboost not installed → skipped (pip install catboost to add it)


## 2. Client-grouped cross-validation (3 × 5 folds)

In [3]:
records, timing = [], {}
for seed in (0, 1, 2):
    folds = GroupKFold(n_splits=5, shuffle=True, random_state=seed).split(episodes, groups=groups)
    for fold, (tr, te) in enumerate(folds):
        for name, fit_predict in MODELS.items():
            t0 = time.time()
            scores = fit_predict(tr, te)
            timing[name] = timing.get(name, 0) + time.time() - t0
            for j, i in enumerate(te):
                s = scores[j].copy()
                s[VISIBLE[i]] = -np.inf
                records.append({"model": name, "seed": seed, "fold": fold, "episode": i,
                                "rank": rank_of_true(scores[j], y[i], VISIBLE[i]),
                                "warm": bool(VISIBLE[i].any()), "long_tail": y[i] not in TOP3,
                                "top3": tuple(np.argsort(-s)[:3])})
R = pd.DataFrame(records)
print(f"{len(R):,} ranked episodes · seconds per model:", {k: round(v, 1) for k, v in timing.items()})

7,920 ranked episodes · seconds per model: {'Popularity': 0.0, 'Segment popularity': 0.0, 'Co-occurrence': 0.0, 'Logistic regression': 0.8, 'Random forest': 9.7, 'Gradient boosting': 17.1}


In [4]:
def summarise(R):
    per_fold = R.groupby(["model", "seed", "fold"])["rank"].apply(lambda r: pd.Series(ranking_report(r))).unstack()
    table = per_fold.groupby("model").agg(["mean", "std"])
    out = pd.DataFrame({m: table[(m, "mean")].map("{:.3f}".format) + " ± " + table[(m, "std")].map("{:.3f}".format)
                        for m in ["HitRate@1", "HitRate@3", "NDCG@3", "MRR"]})
    hit = R.assign(hit3=R["rank"] <= 3)
    out["HitRate@3 warm"] = hit[hit.warm].groupby("model")["hit3"].mean().round(3)
    out["HitRate@3 cold-start"] = hit[~hit.warm].groupby("model")["hit3"].mean().round(3)
    out["HitRate@3 long-tail"] = hit[hit.long_tail].groupby("model")["hit3"].mean().round(3)
    out["Coverage (products)"] = R.groupby("model")["top3"].apply(lambda t: len({p for tt in t for p in tt}))
    out["_sort"] = table[("HitRate@3", "mean")]
    return out.sort_values("_sort", ascending=False).drop(columns="_sort"), per_fold


random_hit3 = np.mean([min(3, len(PRODUCTS) - v.sum()) / (len(PRODUCTS) - v.sum()) for v in VISIBLE])
results, per_fold = summarise(R)
results.loc["Random (expected)"] = ["–", f"{random_hit3:.3f}", "–", "–", np.nan, np.nan, np.nan, np.nan]
results.to_csv(ROOT / "reports" / "model_comparison.csv")
results

,HitRate@1,HitRate@3,NDCG@3,MRR,HitRate@3 warm,HitRate@3 cold-start,HitRate@3 long-tail,Coverage (products)
model,,,,,,,,
Logistic regression,0.388 ± 0.050,0.686 ± 0.032,0.563 ± 0.037,0.571 ± 0.037,0.654,0.72,0.329,17
Random forest,0.392 ± 0.051,0.670 ± 0.035,0.555 ± 0.039,0.566 ± 0.039,0.652,0.688,0.322,24
Segment popularity,0.338 ± 0.059,0.669 ± 0.040,0.526 ± 0.046,0.526 ± 0.047,0.627,0.715,0.222,12
Gradient boosting,0.345 ± 0.050,0.666 ± 0.044,0.533 ± 0.042,0.537 ± 0.039,0.638,0.694,0.306,27
Co-occurrence,0.384 ± 0.053,0.665 ± 0.069,0.544 ± 0.059,0.554 ± 0.048,0.665,0.662,0.16,16
Popularity,0.375 ± 0.072,0.654 ± 0.068,0.532 ± 0.065,0.544 ± 0.058,0.645,0.662,0.107,7
Random (expected),–,0.067,–,–,NaN,NaN,NaN,NaN


### Gain over popularity: paired bootstrap by company

Average each episode’s paired hit difference over the three CV repeats, then resample entire companies with all their episodes. This accounts for dependence within a company. Intervals describe this reconstruction benchmark, conditional on the existing preprocessing and selected model; they do not measure business uplift.


In [5]:
def paired_gain(model_a, model_b="Popularity", metric_mask=None, n_boot=2000, seed=0):
    hits = (R.assign(hit=R["rank"] <= 3).pivot_table(index=["seed", "episode"], columns="model", values="hit")
              .groupby("episode").mean())
    if metric_mask is not None:
        hits = hits.loc[hits.index.isin(np.flatnonzero(metric_mask))]
    diff = (hits[model_a] - hits[model_b]).to_numpy()
    client_groups = groups[hits.index.to_numpy(dtype=int)]
    low, high = cluster_bootstrap_ci(diff, groups=client_groups, n_boot=n_boot, seed=seed)
    return pd.Series({"gain (pts)": diff.mean() * 100, "95% CI low": low * 100,
                      "95% CI high": high * 100, "episodes": len(diff), "companies": len(np.unique(client_groups))}).round(1)


long_tail_mask = np.array([t not in TOP3 for t in y])
gains = pd.DataFrame({
    "LogReg vs Popularity · all": paired_gain("Logistic regression"),
    "LogReg vs Popularity · long-tail": paired_gain("Logistic regression", metric_mask=long_tail_mask),
    "RandomForest vs Popularity · long-tail": paired_gain("Random forest", metric_mask=long_tail_mask),
}).T
gains
# Export the uncertainty evidence used by the README and presentation.
gains.to_csv(ROOT / "reports" / "model_gain_company_bootstrap.csv")


## 3. Visual comparison

In [6]:
order = (per_fold.groupby("model")["HitRate@3"].mean().sort_values())
err = per_fold.groupby("model")["HitRate@3"].std().reindex(order.index)
best = order.idxmax()
fig, ax = plt.subplots(figsize=(9, 4.6))
colors = [ORANGE if m == best else (INK_3 if m in ("Popularity", "Segment popularity") else BLUE) for m in order.index]
ax.barh(order.index, order.values, xerr=err.values, color=colors, height=0.6,
        error_kw=dict(ecolor=INK_2, lw=1, capsize=3))
for i, (v, e) in enumerate(zip(order.values, err.values)):
    ax.text(v + e + 0.01, i, f"{v:.3f}", va="center", fontsize=9, color=INK_2)
ax.axvline(random_hit3, color=INK, lw=1, ls="--")
ax.text(random_hit3 + 0.008, -0.62, f"random = {random_hit3:.2f}", fontsize=8.5, color=INK, va="center")
ax.set_ylim(-0.9, len(order) - 0.5)
ax.set_xlim(0, 0.9)
ax.set_xlabel("HitRate@3 (mean ± std over 15 client-grouped folds)")
ax.set_title("All models find the hidden product in the top-3 about 2 times in 3", pad=26)
subtitle(ax, "Grey = business rules · blue = learned models · orange = best · popularity is a strong baseline in a motor-dominated portfolio")
ax.grid(axis="y", visible=False)
save(fig, "fig13_model_comparison")
plt.show()

In [7]:
show = ["Popularity", "Segment popularity", "Co-occurrence", "Logistic regression", "Random forest"]
lt = results.loc[show, "HitRate@3 long-tail"].astype(float)
cov = results.loc[show, "Coverage (products)"].astype(float)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.75})
for ax, series, title, sub, fmt in [
    (axes[0], lt, "Long-tail HitRate@3", "Products beyond the 3 best-sellers: learned models ×3", "{:.2f}"),
    (axes[1], cov, "Coverage: products ever recommended", "Learned models personalise; popularity repeats 7 products", "{:.0f}"),
]:
    c = [INK_3 if m in ("Popularity", "Segment popularity") else BLUE for m in series.index]
    ax.barh(series.index, series.values, color=c, height=0.6)
    for i, v in enumerate(series.values):
        ax.text(v * 1.02 + 0.002, i, fmt.format(v), va="center", fontsize=9, color=INK_2)
    ax.set_title(title, pad=24)
    subtitle(ax, sub)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
save(fig, "fig14_longtail_coverage")
plt.show()

**Reading.** On raw HitRate@3 every approach lands around 0.65–0.69, because 6 companies in 10
hold a motor product: proposing motor is right most of the time. The difference is elsewhere:
learned models recover **non-obvious products about 3× more often** and spread their
recommendations over many more products. For BO1, diversification matters. The baseline excludes exact owned products but does not exclude other products in the same insurance line, so an automobile variant can still rank first; long-tail performance alone does not establish cross-line diversification. DSO1.1 therefore keeps two targets (see notebook 01):
**HitRate@3 ≥ 0.70 overall and ≥ 0.40 on the long tail.**

## 4. What drives the predictions? (random forest, fitted on all episodes)

In [8]:
final_rf = make_pipeline(build_preprocessor(PRODUCTS),
                         RandomForestClassifier(n_estimators=500, min_samples_leaf=2, random_state=0, n_jobs=-1))
final_rf.fit(episodes, y)
names = final_rf[0].get_feature_names_out()
imp = pd.Series(final_rf[-1].feature_importances_, index=names).sort_values(ascending=False)

def pretty(n):
    n = n.split("__", 1)[1]
    return (n.replace("has::", "holds · ").replace("activity_freq", "activity (frequency-encoded)").replace("category_", "category = ").replace("sector_", "sector = ")
             .replace("OMNICANAL AUTOMOBILE ", "OMNI AUTO ").replace("MULTIRISQUES PROFESSIONNELLES", "MULTIRISQUE PRO"))[:46]

top = imp.head(12)[::-1]
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh([pretty(n) for n in top.index], top.values, color=BLUE, height=0.6)
ax.set_xlabel("mean decrease in impurity")
ax.set_title("Activity, portfolio and sector drive the recommendation", pad=26)
subtitle(ax, "Top-12 features of the random forest · the profile matters as much as what is already held")
ax.grid(axis="y", visible=False)
save(fig, "fig15_feature_importance")
plt.show()

*Caveat:* impurity importance favours continuous features such as activity frequency. Ranking-based permutation importance is future work.

## 5. Local recommendation tool demonstration (DSO1.3)
The function below and its schema demonstrate a possible interface for the proposed cross-sell agent. This notebook does **not** demonstrate an actual LLM tool call, a deployed endpoint, retrieval-grounded drafting or an end-to-end advisor workflow. New API contracts can differ from the prototype.

Scores are relative model scores renormalised over unheld products, **not calibrated probabilities of future purchase**. Excluding exact products does not guarantee diversification across insurance lines. The example is retained to expose that baseline limitation rather than select a more flattering case.

The company-bootstrap interval for LR versus popularity is **+0.1 to +6.5 points overall** and **+15.8 to +29.6 points on long-tail products** (2,000 resamples; seed 0). The overall interval narrowly excludes zero, conditional on preprocessing and model selection. Neither interval establishes future conversion uplift.


In [9]:
TOOL_SCHEMA = {
    "name": "recommend_products",
    "description": ("Rank k unheld insurance products for a corporate client of BH Assurance using relative model scores; "
                    "return the insurance line. Scores are not calibrated purchase probabilities."),
    "input_schema": {
        "type": "object",
        "properties": {
            "client_id": {"type": "integer", "description": "REF_pm of the company"},
            "k": {"type": "integer", "default": 3, "minimum": 1, "maximum": 5},
        },
        "required": ["client_id"],
    },
}
print(json.dumps(TOOL_SCHEMA, indent=2, ensure_ascii=False))

{
  "name": "recommend_products",
  "description": "Rank k unheld insurance products for a corporate client of BH Assurance using relative model scores; return the insurance line. Scores are not calibrated purchase probabilities.",
  "input_schema": {
    "type": "object",
    "properties": {
      "client_id": {
        "type": "integer",
        "description": "REF_pm of the company"
      },
      "k": {
        "type": "integer",
        "default": 3,
        "minimum": 1,
        "maximum": 5
      }
    },
    "required": [
      "client_id"
    ]
  }
}


In [10]:
final_lr = make_pipeline(build_preprocessor(PRODUCTS), LogisticRegression(C=0.3, max_iter=3000)).fit(episodes, y)
companies = pd.read_csv(PROCESSED / "pm_companies_clean.csv").fillna({"portfolio": ""})


def recommend_products(client_id: int, k: int = 3) -> dict:
    """Local inference demo with the full current portfolio; not a deployed agent tool."""
    c = companies.loc[companies["REF_pm"] == client_id].iloc[0]
    held = [p for p in c["portfolio"].split(";") if p]
    row = pd.DataFrame([{**c.to_dict(), "visible_portfolio": ";".join(held), "n_visible": len(held)}])
    proba = np.zeros(len(PRODUCTS))
    proba[final_lr.classes_] = final_lr.predict_proba(row)[0]
    proba[[IDX[p] for p in held]] = 0.0
    proba = proba / proba.sum()                        # relative model score among unheld products, not purchase probability
    best = np.argsort(-proba)[:k]
    return {"client_id": int(client_id), "category": c["category"], "sector": c["sector"], "held": held,
            "recommendations": [{"product": PRODUCTS[i], "line": catalogue["product_branch"].get(PRODUCTS[i], "n/a"),
                                 "score": round(float(proba[i]), 3)} for i in best],
            "model": "logreg-v0.1 (validation 1 baseline)"}


example = recommend_products(29893)      # mono-product construction company (house builder)
print(json.dumps(example, indent=2, ensure_ascii=False))

{
  "client_id": 29893,
  "category": "CONSTRUCTION & REAL ESTATE",
  "sector": "CONSTRUCTION",
  "held": [
    "AUTOMOBILE"
  ],
  "recommendations": [
    {
      "product": "AUTOMOBILE II",
      "line": "AUTOMOBILE",
      "score": 0.495
    },
    {
      "product": "POLICE ABONNEMENT(FACULTE MARITIME)",
      "line": "TRANSPORT",
      "score": 0.147
    },
    {
      "product": "MULTIRISQUES PROFESSIONNELLES CENTRALISE",
      "line": "INCENDIE",
      "score": 0.082
    }
  ],
  "model": "logreg-v0.1 (validation 1 baseline)"
}


In [11]:
print(json.dumps(recommend_products(12112), indent=2, ensure_ascii=False))   # company of the prototype's recommend_pm sample
_ = (ROOT / "reports" / "tool_output_example.json").write_text(json.dumps(
    {"tool_schema": TOOL_SCHEMA, "example_call": {"client_id": 29893, "k": 3}, "example_output": example},
    indent=2, ensure_ascii=False))

{
  "client_id": 12112,
  "category": "FINANCE & ASSURANCE",
  "sector": "INTERMEDIATION FINANCIERE",
  "held": [
    "AUTOMOBILE",
    "AUTOMOBILE II",
    "AUTOMOBILE LEASING",
    "INDIVIDUELLE ACCIDENTS",
    "MULTIRISQUES INFORMATIQUES",
    "MULTIRISQUES PROFESSIONNELLES CENTRALISE",
    "RESPONSABILITE CIVILE"
  ],
  "recommendations": [
    {
      "product": "POLICE AU VOYAGE(FACULTE TERRESTRE)",
      "line": "TRANSPORT",
      "score": 0.168
    },
    {
      "product": "POLICE ABONNEMENT(FACULTE MARITIME)",
      "line": "TRANSPORT",
      "score": 0.126
    },
    {
      "product": "ENGINS DE CHANTIERS",
      "line": "ENGINEERING",
      "score": 0.118
    }
  ],
  "model": "logreg-v0.1 (validation 1 baseline)"
}


## 6. Modeling approach for every DSO and current status

In [12]:
plan = pd.DataFrame([
    ["DSO1.1", "Next-product propensity", "Multinomial LR / RF / CatBoost on leave-one-out episodes", "HitRate@3, long-tail HitRate@3, coverage", "Evaluated v1: 0.686 / 0.329; v2 implemented, not evaluated here"],
    ["DSO1.2", "Client segmentation", "K-means on binary category, insurance lines, portfolio depth and history", "Silhouette, segment size, business review", "Implemented in src/segments.py; not evaluated here"],
    ["DSO1.3", "Reco tool (agentic)", "FastAPI endpoint + function-calling schema + RAG justification", "% valid tool calls on 50 scenarios", "Local function + schema demonstrated; integrated agent proposed"],
    ["DSO2.1", "Non-payment risk", "Logistic / gradient boosting with class weights", "Recall on unpaid, PR-AUC", "⛔ 12 positives → more history requested"],
    ["DSO2.2", "Guarantee upsell", "Apriori-style rule enumeration on guarantees + product/sub-branch context", "Lift, confidence, coverage", "Implemented in src/rules.py; not evaluated here"],
    ["DSO2.3", "Renewal agent (agentic)", "LangGraph state machine: remind / offer / escalate + tool calls", "% contracts contacted ≥ 30 days before expiry", "⏳ design"],
    ["DSO3.1", "Intent classifier", "Prototype regex baseline + TF-IDF word/character n-grams and logistic regression", "Macro-F1 on 300 labelled messages", "Implemented in src/nlp.py; real-message evaluation pending"],
    ["DSO3.2", "RAG on general conditions", "BM25 / multilingual-e5-small / reciprocal-rank fusion", "Recall@5, faithfulness", "Implemented in src/nlp.py; evaluation not demonstrated here"],
    ["DSO3.3", "Orchestrator (agentic)", "LangChain/LangGraph agent with tools + human validation", "Approval-without-edit rate", "Proposed new orchestration; inherited prototype is starting point"],
    ["DSO4.1", "Value / loyalty score", "Weighted score + ordinal model", "Spearman vs future holdings", "⏳ planned"],
    ["DSO4.2", "LLM-as-judge QA", "Rubric-based LLM evaluation of drafts", "Cohen's κ vs reviewers", "⏳ planned"],
    ["DSO4.3", "Monitoring agent (agentic)", "Agent reading MongoDB logs → KPI tables → dashboard", "Daily refresh, weekly report", "⏳ planned"],
], columns=["DSO", "Objective", "Approach", "Metric", "Status (validation 1)"])
plan

,DSO,Objective,Approach,Metric,Status (validation 1)
0,DSO1.1,Next-product propensity,Multinomial LR / RF / CatBoost on leave-one-ou...,"HitRate@3, long-tail HitRate@3, coverage","Evaluated v1: 0.686 / 0.329; v2 implemented, n..."
1,DSO1.2,Client segmentation,"K-means on binary category, insurance lines, p...","Silhouette, segment size, business review",Implemented in src/segments.py; not evaluated ...
2,DSO1.3,Reco tool (agentic),FastAPI endpoint + function-calling schema + R...,% valid tool calls on 50 scenarios,Local function + schema demonstrated; integrat...
3,DSO2.1,Non-payment risk,Logistic / gradient boosting with class weights,"Recall on unpaid, PR-AUC",⛔ 12 positives → more history requested
4,DSO2.2,Guarantee upsell,Apriori-style rule enumeration on guarantees +...,"Lift, confidence, coverage",Implemented in src/rules.py; not evaluated here
5,DSO2.3,Renewal agent (agentic),LangGraph state machine: remind / offer / esca...,% contracts contacted ≥ 30 days before expiry,⏳ design
6,DSO3.1,Intent classifier,Prototype regex baseline + TF-IDF word/charact...,Macro-F1 on 300 labelled messages,Implemented in src/nlp.py; real-message evalua...
7,DSO3.2,RAG on general conditions,BM25 / multilingual-e5-small / reciprocal-rank...,"Recall@5, faithfulness",Implemented in src/nlp.py; evaluation not demo...
8,DSO3.3,Orchestrator (agentic),LangChain/LangGraph agent with tools + human v...,Approval-without-edit rate,Proposed new orchestration; inherited prototyp...
9,DSO4.1,Value / loyalty score,Weighted score + ordinal model,Spearman vs future holdings,⏳ planned


## 7. Next steps toward validation 2
1. Move learned imputation into grouped CV and audit history-feature availability; tune with nested grouped CV.
2. Evaluate the implemented v2 recommender, segmentation, guarantee rules and NLP modules; implementation alone is not a validated result.
3. Define cross-line eligibility and suitability rules with business reviewers; calibrate only against an appropriate labelled outcome.
4. Integrate cross-sell and renewal agents, distinct segmentation/value tools, guarantee recommendations, draft quality and monitoring under the proposed orchestrator. Keep advisor validation mandatory.
5. Evaluate actual tool calls, retrieval, draft quality and business outcomes with their own benchmarks; preserve the v1 baseline for comparison.

**Presentation status:** baseline targets are not yet reached (0.686 vs 0.70 overall; 0.329 vs 0.40 long-tail). Team labels remain placeholders until the four names are supplied. Renewal experiments are historical simulations, not live operational results.
